<a href="https://colab.research.google.com/github/davidlealo/proyecto_sysrec_2026_1/blob/main/benchmark_hw_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GreenRecSys — Benchmark por Hardware

Se usó IA para documentar el código, conversación en: https://gemini.google.com/share/a881d782fcf6

### 1. Resolución de Dependencias y Entorno Base
Google Colab preinstala versiones recientes de librerías (como NumPy >= 2.0) que presentan conflictos de compatibilidad con las dependencias binarias y el código heredado de RecBole.

**Decisión técnica:** Se fuerza el "downgrade" de NumPy a la versión `1.26.4` y se limitan las versiones de PyTorch, SciPy y Pandas. Esto garantiza un entorno estable y predecible antes de instalar el resto del stack, evitando errores de ejecución en el motor de evaluación. *Nota: Es necesario reiniciar el runtime después de esta ejecución.*

In [1]:
# Colab trae NumPy >= 2.0, que es incompatible con RecBole.
# Se fuerza 1.26.4 y luego se reinicia el runtime antes de continuar.
!pip install -q "numpy==1.26.4" --force-reinstall

import numpy as np
assert np.__version__.startswith("1."), (
    f"NumPy sigue siendo {np.__version__}. "
    "Reiniciar el runtime (Runtime > Restart session) y volver a ejecutar desde esta celda."
)
print(f"numpy {np.__version__}")

!pip install -q "torch<2.6" "scipy<1.13" "pandas<2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you

Otras librerías que tuve que cargarlas después porque antes no funcionaba

### 2. Instalación del Stack Principal
**Decisión técnica:** Se instalan las librerías necesarias para la ejecución del benchmark. Destacan **RecBole** como framework unificado para los algoritmos de recomendación y **CodeCarbon** como herramienta principal para la medición de la huella de carbono y el consumo energético (GreenRecSys). También se incluyen librerías de soporte algorítmico como `lightgbm`, `xgboost` y `kmeans_pytorch`.

In [2]:
!pip install -q recbole kmeans_pytorch ray lightgbm xgboost codecarbon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.5/380.5 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 186.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 190.7 MB/s eta 0:00:00


### 3. Parches de Compatibilidad (Monkey-Patching) en Tiempo de Ejecución

Se identificaron incompatibilidades críticas entre las versiones actuales de los paquetes del entorno y RecBole 1.2.1. En lugar de modificar el código fuente de la librería (lo que dificultaría la reproducibilidad), se aplican parches dinámicos e idempotentes (se pueden ejecutar múltiples veces en el mismo runtime sin generar recursión ni doble aplicación).

**Decisiones técnicas:**

* **(a) Compatibilidad con PyTorch >= 2.6:** La nueva versión de PyTorch cambió el comportamiento por defecto de `torch.load` a `weights_only=True`. RecBole llama a esta función sin dicho argumento al cargar *checkpoints*, lo que lanza un `UnpicklingError`. El parche intercepta la llamada original para forzar `weights_only=False`.
* **(b) Refactor de Matriz de Adyacencia (SciPy >= 1.14):** Las versiones recientes de SciPy eliminaron `dok_matrix._update` y bloquearon `dok_matrix.update`. RecBole utiliza estos métodos en arquitecturas basadas en grafos (LightGCN y NGCF) para construir la matriz de adyacencia. Se reemplaza el método original `get_norm_adj_mat` por una versión que utiliza `csr_matrix` y devuelve directamente un *torch sparse tensor*, cumpliendo con lo que el motor de RecBole espera.

In [3]:
import functools
import torch
import scipy.sparse as sp
import numpy as np
from recbole.model.general_recommender import lightgcn, ngcf

# torch.load
if not getattr(torch.load, "_recbole_patched", False):
    _orig_torch_load = torch.load

    @functools.wraps(_orig_torch_load)
    def _patched_torch_load(f, map_location=None, pickle_module=None, weights_only=True, **kwargs):
        return _orig_torch_load(f, map_location=map_location, weights_only=False, **kwargs)

    _patched_torch_load._recbole_patched = True
    torch.load = _patched_torch_load

# LightGCN / NGCF
def _get_norm_adj_mat_fixed(self):
    user_np = self.interaction_matrix.row
    item_np = self.interaction_matrix.col
    ratings = np.ones(len(user_np))
    n_user  = self.n_users
    n_item  = self.n_items

    tmp_adj = sp.csr_matrix(
        (ratings, (user_np, item_np + n_user)),
        shape=(n_user + n_item, n_user + n_item),
    )
    adj      = tmp_adj + tmp_adj.T
    rowsum   = np.array(adj.sum(axis=1)).flatten()
    d_inv_sq = np.power(rowsum, -0.5, where=rowsum > 0, out=np.zeros_like(rowsum, dtype=float))
    norm_adj = (sp.diags(d_inv_sq) @ adj @ sp.diags(d_inv_sq)).tocoo().astype(np.float32)

    indices = torch.from_numpy(np.vstack([norm_adj.row, norm_adj.col]).astype(np.int64))
    values  = torch.from_numpy(norm_adj.data)
    return torch.sparse_coo_tensor(indices, values, torch.Size(norm_adj.shape))

_get_norm_adj_mat_fixed._recbole_patched = True

if not getattr(lightgcn.LightGCN.get_norm_adj_mat, "_recbole_patched", False):
    lightgcn.LightGCN.get_norm_adj_mat = _get_norm_adj_mat_fixed

try:
    if not getattr(ngcf.NGCF.get_norm_adj_mat, "_recbole_patched", False):
        ngcf.NGCF.get_norm_adj_mat = _get_norm_adj_mat_fixed
except Exception:
    pass

print("parches aplicados")

parches aplicados


### 4. Detección Dinámica de Hardware
Para que el benchmark sea válido, es fundamental registrar el contexto físico donde se ejecutan los modelos.

**Decisión técnica:** Se implementa un script que consulta al sistema operativo y a la API de CUDA para extraer la topología del hardware subyacente (modelo de CPU, GPU, VRAM disponible y RAM del sistema). Esto permite etiquetar automáticamente los resultados exportados, facilitando el análisis comparativo posterior entre distintos runtimes (CPU vs. T4 vs. A100, etc.).

In [11]:
import subprocess
import platform

def detectar_hardware():
    info = {}

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        info["gpu"]     = gpu_name
        info["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
        info["device"]  = "cuda"

        g = gpu_name.lower()
        if   "h100" in g:            hw_label = "H100_GPU"
        elif "a100" in g:            hw_label = "A100_GPU"
        elif "l4"   in g:            hw_label = "L4_GPU"
        elif "t4"   in g:            hw_label = "T4_GPU"
        elif "g4"   in g or "v100" in g: hw_label = "G4_GPU"
        else:                        hw_label = gpu_name.replace(" ", "_")
    else:
        info["gpu"]    = None
        info["device"] = "cpu"
        try:
            import torch_xla
            hw_label       = "TPU"
            info["device"] = "xla"
        except ImportError:
            hw_label = "CPU"

    try:
        raw = subprocess.check_output("lscpu | grep 'Model name'", shell=True).decode().strip()
        info["cpu"] = raw.split(":")[-1].strip()
    except Exception:
        info["cpu"] = platform.processor() or "desconocido"

    try:
        raw   = subprocess.check_output("free -g | grep Mem", shell=True).decode().strip()
        ram   = int(raw.split()[1])
        info["ram_gb"]  = ram
        info["high_ram"] = ram >= 50
        if info["high_ram"]:
            hw_label += "_HighRAM"
    except Exception:
        info["ram_gb"]  = None
        info["high_ram"] = False

    info["hw_label"] = hw_label
    return info


HW_INFO  = detectar_hardware()
HW_LABEL = HW_INFO["hw_label"]
DEVICE   = HW_INFO["device"]

print("-" * 46)
print(f"hardware : {HW_LABEL}")
print(f"device   : {DEVICE}")
print(f"cpu      : {HW_INFO.get('cpu', 'N/A')}")
print(f"ram      : {HW_INFO.get('ram_gb', 'N/A')} GB  (high_ram={HW_INFO.get('high_ram', False)})")
if HW_INFO.get("gpu"):
    print(f"gpu      : {HW_INFO['gpu']} ({HW_INFO.get('vram_gb', '?')} GB VRAM)")
print("-" * 46)

----------------------------------------------
hardware : NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_HighRAM
device   : cuda
cpu      : AMD EPYC 9B45
ram      : 176 GB  (high_ram=True)
gpu      : NVIDIA RTX PRO 6000 Blackwell Server Edition (102.0 GB VRAM)
----------------------------------------------


### 5. Diseño Experimental y Grilla de Hiperparámetros
Se define la estructura metodológica del experimento.

**Decisiones técnicas:**
* **Métricas y Evaluación:** Se establecen métricas estándar para recomendadores (NDCG, Recall, MRR, etc.) con cortes en Top-5 y Top-10. La partición de datos se fija en 80/10/10.
* **Control de Variables:** Se generan diccionarios de configuración iterando sobre diferentes combinaciones de hiperparámetros (épocas, learning rate, tamaño de batch, embedding size).
* **Reproducibilidad:** Se configuran semillas específicas (2024, 2025, 2026) para los modelos probabilísticos y funciones de pérdida (BPR, CE, MSE) para los modelos que lo soportan, asegurando la rigurosidad y trazabilidad del benchmark.

In [5]:
import warnings
import pandas as pd
from logging import getLogger
from codecarbon import EmissionsTracker
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger, get_model, get_trainer

warnings.filterwarnings("ignore")

dataset_name = "ml-100k"

base_config = {
    "seed": 2024,
    "reproducibility": True,
    "device": DEVICE if DEVICE != "xla" else "cpu",
    "metrics": ["NDCG", "Recall", "MRR", "Precision", "Hit", "MAP"],
    "topk": [5, 10],
    "valid_metric": "NDCG@10",
    "eval_args": {"split": {"RS": [0.8, 0.1, 0.1]}, "order": "RO", "mode": "full"},
}

loss_options  = ["BPR", "CE", "MSE"]
loss_models   = {"BPR", "LightGCN", "NeuMF", "NGCF"}
seed_variants = [2024, 2025, 2026]


def make_loss_configs(model_name, base_runs):
    configs = []
    for loss in loss_options:
        for idx, run in enumerate(base_runs, start=1):
            cfg = run.copy()
            cfg["loss_type"] = loss
            configs.append({"name": f"{model_name}_{loss}_r{idx}", "config": cfg, "loss": loss})
    return configs


def make_noloss_configs(model_name, base_runs):
    configs = []
    for seed in seed_variants:
        for idx, run in enumerate(base_runs, start=1):
            cfg = run.copy()
            cfg["seed"] = seed
            configs.append({"name": f"{model_name}_s{seed}_r{idx}", "config": cfg, "loss": "N/A"})
    return configs


base_runs = {
    "BPR": [
        {"epochs": 4, "learning_rate": 0.002,  "embedding_size": 16, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.001,  "embedding_size": 8,  "train_batch_size": 2048},
        {"epochs": 2, "learning_rate": 0.003,  "embedding_size": 12, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.0015, "embedding_size": 24, "train_batch_size": 1024},
        {"epochs": 2, "learning_rate": 0.002,  "embedding_size": 20, "train_batch_size": 2048},
        {"epochs": 2, "learning_rate": 0.001,  "embedding_size": 12, "train_batch_size": 4096},
    ],
    "LightGCN": [
        {"epochs": 6, "learning_rate": 0.001,  "embedding_size": 16, "n_layers": 2, "reg_weight": 1e-5, "train_batch_size": 1024},
        {"epochs": 5, "learning_rate": 0.0008, "embedding_size": 8,  "n_layers": 1, "reg_weight": 1e-4, "train_batch_size": 2048},
        {"epochs": 4, "learning_rate": 0.001,  "embedding_size": 12, "n_layers": 1, "reg_weight": 1e-5, "train_batch_size": 1024},
        {"epochs": 4, "learning_rate": 0.0012, "embedding_size": 20, "n_layers": 2, "reg_weight": 1e-5, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.0007, "embedding_size": 10, "n_layers": 1, "reg_weight": 5e-5, "train_batch_size": 2048},
        {"epochs": 3, "learning_rate": 0.001,  "embedding_size": 14, "n_layers": 1, "reg_weight": 1e-5, "train_batch_size": 4096},
    ],
    "FM": [
        {"epochs": 4, "learning_rate": 0.002,  "embedding_size": 16, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.001,  "embedding_size": 8,  "train_batch_size": 2048},
        {"epochs": 2, "learning_rate": 0.003,  "embedding_size": 12, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.0015, "embedding_size": 20, "train_batch_size": 1024},
        {"epochs": 2, "learning_rate": 0.002,  "embedding_size": 24, "train_batch_size": 2048},
        {"epochs": 2, "learning_rate": 0.001,  "embedding_size": 10, "train_batch_size": 4096},
    ],
    "NeuMF": [
        {"epochs": 4, "learning_rate": 0.001,  "embedding_size": 16, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.0008, "embedding_size": 8,  "train_batch_size": 2048},
        {"epochs": 2, "learning_rate": 0.0015, "embedding_size": 12, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.0012, "embedding_size": 20, "train_batch_size": 1024},
        {"epochs": 2, "learning_rate": 0.001,  "embedding_size": 24, "train_batch_size": 2048},
        {"epochs": 2, "learning_rate": 0.0007, "embedding_size": 10, "train_batch_size": 4096},
    ],
    "NGCF": [
        {"epochs": 5, "learning_rate": 0.001,  "embedding_size": 16, "n_layers": 2, "reg_weight": 1e-5, "train_batch_size": 1024},
        {"epochs": 4, "learning_rate": 0.0008, "embedding_size": 8,  "n_layers": 1, "reg_weight": 1e-4, "train_batch_size": 2048},
        {"epochs": 3, "learning_rate": 0.0012, "embedding_size": 12, "n_layers": 1, "reg_weight": 1e-5, "train_batch_size": 1024},
        {"epochs": 3, "learning_rate": 0.001,  "embedding_size": 20, "n_layers": 2, "reg_weight": 1e-5, "train_batch_size": 1024},
        {"epochs": 2, "learning_rate": 0.0007, "embedding_size": 10, "n_layers": 1, "reg_weight": 5e-5, "train_batch_size": 2048},
        {"epochs": 2, "learning_rate": 0.001,  "embedding_size": 14, "n_layers": 1, "reg_weight": 1e-5, "train_batch_size": 4096},
    ],
    "Pop": [
        {"epochs": 1, "train_batch_size": 4096},
        {"epochs": 1, "train_batch_size": 2048},
        {"epochs": 1, "train_batch_size": 1024},
        {"epochs": 1, "train_batch_size": 512},
        {"epochs": 1, "train_batch_size": 256},
        {"epochs": 1, "train_batch_size": 128},
    ],
    "ItemKNN": [
        {"epochs": 1, "k": 20,  "shrink": 10, "train_batch_size": 2048},
        {"epochs": 1, "k": 40,  "shrink": 20, "train_batch_size": 1024},
        {"epochs": 1, "k": 80,  "shrink": 50, "train_batch_size": 512},
        {"epochs": 1, "k": 30,  "shrink": 5,  "train_batch_size": 2048},
        {"epochs": 1, "k": 60,  "shrink": 30, "train_batch_size": 1024},
        {"epochs": 1, "k": 100, "shrink": 80, "train_batch_size": 512},
    ],
    "Random": [
        {"epochs": 1, "train_batch_size": 4096},
        {"epochs": 1, "train_batch_size": 2048},
        {"epochs": 1, "train_batch_size": 1024},
        {"epochs": 1, "train_batch_size": 512},
        {"epochs": 1, "train_batch_size": 256},
        {"epochs": 1, "train_batch_size": 128},
    ],
}

benchmark_configs = {}
for model_name, runs in base_runs.items():
    if model_name in loss_models:
        benchmark_configs[model_name] = make_loss_configs(model_name, runs)
    else:
        benchmark_configs[model_name] = make_noloss_configs(model_name, runs)

total_runs = sum(len(v) for v in benchmark_configs.values())
print(f"configuraciones a ejecutar: {total_runs}")

configuraciones a ejecutar: 144


### 6. Orquestación del Entrenamiento y Medición de Emisiones
Se encapsula la lógica de inicialización, entrenamiento y evaluación de RecBole.

**Decisiones técnicas:**
* **Integración con CodeCarbon:** Se envuelve el ciclo de `trainer.fit()` y `trainer.evaluate()` dentro de un contexto de `EmissionsTracker` (configurado a 1 segundo de resolución) para capturar el consumo energético exacto de cada ejecución.
* **Manejo de Excepciones de Loss:** Si un modelo particular no soporta una función de pérdida solicitada en la grilla, el bloque `try-except` captura el error de validación, descarta el parámetro conflictivo y reintenta la ejecución, registrando esta anomalía para no perder la iteración.

In [6]:
def _run_with_config(model_name, config_dict):
    config = Config(model=model_name, dataset=dataset_name, config_dict=config_dict)
    init_seed(config["seed"], config["reproducibility"])
    init_logger(config)
    getLogger().info(config)

    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)
    model   = get_model(config["model"])(config, train_data.dataset).to(config["device"])
    trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

    tracker = EmissionsTracker(
        project_name="greenrecsys_hw_benchmark",
        measure_power_secs=1,
        log_level="error",
        save_to_file=False,
    )
    tracker.start()
    trainer.fit(train_data, valid_data)
    test_result  = trainer.evaluate(test_data)
    emissions_kg = tracker.stop()
    return test_result, emissions_kg


def run_experiment(model_name, config_overrides):
    """
    Ejecuta un experimento. Si el modelo no soporta el loss_type solicitado,
    reintenta sin ese parametro y registra loss_used como None.
    """
    config_dict = {**base_config, **config_overrides}
    loss_used   = config_dict.get("loss_type")
    try:
        test_result, emissions_kg = _run_with_config(model_name, config_dict)
    except ValueError as exc:
        if loss_used and "loss" in str(exc).lower():
            config_dict = {k: v for k, v in config_dict.items() if k != "loss_type"}
            loss_used   = None
            test_result, emissions_kg = _run_with_config(model_name, config_dict)
        else:
            raise
    return test_result, emissions_kg, loss_used

### 7. Bucle de Ejecución del Benchmark
Esta celda ejecuta secuencialmente las 144 configuraciones generadas.

**Decisiones técnicas:**
* **Tolerancia a fallos:** Dado que los runtimes en la nube pueden interrumpirse, se introduce la variable `SKIP_N`. Si el proceso se corta, esta variable permite retomar el benchmark exactamente desde la iteración donde se detuvo, evitando desperdiciar cómputo (y emisiones) en recalcular modelos ya procesados.
* **Estructuración de Datos:** Los resultados de tiempo, emisiones de CO2 y métricas de evaluación se aplanan e ingieren iterativamente en una lista de diccionarios, manteniéndose en memoria listos para la serialización.

In [7]:
import time

SKIP_N = 0

metric_keys = [
    "ndcg@5", "ndcg@10", "recall@5", "recall@10",
    "mrr@5",  "mrr@10",  "precision@5", "precision@10",
    "hit@5",  "hit@10",  "map@5", "map@10",
    # RecBole puede devolver las claves en mayusculas segun version
    "NDCG@5", "NDCG@10", "Recall@5", "Recall@10",
    "MRR@5",  "MRR@10",  "Precision@5", "Precision@10",
    "Hit@5",  "Hit@10",  "MAP@5", "MAP@10",
]

if "records" not in dir():
    records = []

total   = sum(len(v) for v in benchmark_configs.values())
done    = 0
t_start = time.time()

for model_name, configs in benchmark_configs.items():
    for config_entry in configs:
        done += 1
        if done <= SKIP_N:
            print(f"[{done}/{total}] {config_entry['name']} — omitido")
            continue

        config_name      = config_entry["name"]
        config_overrides = config_entry["config"]
        loss_label       = config_entry.get("loss", "N/A")

        t0 = time.time()
        test_result, emissions_kg, loss_used = run_experiment(model_name, config_overrides)
        elapsed = time.time() - t0

        row = {
            "hardware":  HW_LABEL,
            "device":    DEVICE,
            "gpu_name":  HW_INFO.get("gpu") or "N/A",
            "ram_gb":    HW_INFO.get("ram_gb") or "N/A",
            "high_ram":  HW_INFO.get("high_ram", False),
            "model":     model_name,
            "config":    config_name,
            "loss":      loss_label,
            "loss_used": loss_used if loss_used else "N/A",
            "co2_kg":    emissions_kg,
            "time_s":    round(elapsed, 2),
        }
        for key in metric_keys:
            if key in test_result:
                row[key.lower()] = test_result[key]

        records.append(row)
        print(f"[{done}/{total}] {config_name} | co2: {emissions_kg:.2e} kg | t: {elapsed:.1f}s")

elapsed_total = time.time() - t_start
print(f"\nbenchmark completo en {elapsed_total / 60:.1f} min")

results_df = pd.DataFrame(records).sort_values(["model", "config"]).reset_index(drop=True)
results_df

[codecarbon WARNING @ 00:41:04] Multiple instances of codecarbon are allowed to run at the same time.


[1/144] BPR_BPR_r1 | co2: 2.41e-05 kg | t: 6.8s


[2/144] BPR_BPR_r2 | co2: 9.84e-06 kg | t: 1.0s


[3/144] BPR_BPR_r3 | co2: 7.56e-06 kg | t: 1.1s


[4/144] BPR_BPR_r4 | co2: 1.11e-05 kg | t: 1.1s


...

[141/144] Random_s2026_r3 | co2: 2.80e-06 kg | t: 0.5s


[142/144] Random_s2026_r4 | co2: 3.12e-06 kg | t: 0.8s


[143/144] Random_s2026_r5 | co2: 5.11e-06 kg | t: 0.6s


[144/144] Random_s2026_r6 | co2: 7.01e-06 kg | t: 1.1s

benchmark completo en 3.6 min


,hardware,device,gpu_name,ram_gb,high_ram,model,config,loss,loss_used,co2_kg,...,recall@5,recall@10,mrr@5,mrr@10,precision@5,precision@10,hit@5,hit@10,map@5,map@10
0,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,BPR,BPR_BPR_r1,BPR,BPR,0.000024,...,0.0852,0.1292,0.2926,0.3087,0.1406,0.1144,0.4592,0.5822,0.1025,0.0842
1,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,BPR,BPR_BPR_r2,BPR,BPR,0.000010,...,0.0181,0.0314,0.1038,0.1139,0.0498,0.0439,0.1888,0.2662,0.0285,0.0211
2,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,BPR,BPR_BPR_r3,BPR,BPR,0.000008,...,0.0584,0.1070,0.2496,0.2688,0.1262,0.1085,0.3860,0.5292,0.0907,0.0726
3,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,BPR,BPR_BPR_r4,BPR,BPR,0.000011,...,0.0633,0.1008,0.2765,0.2911,0.1364,0.1127,0.4019,0.5101,0.1028,0.0807
4,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,BPR,BPR_BPR_r5,BPR,BPR,0.000007,...,0.0346,0.0650,0.1810,0.1958,0.0899,0.0814,0.3022,0.4157,0.0572,0.0450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,Random,Random_s2026_r2,N/A,N/A,0.000003,...,0.0028,0.0067,0.0172,0.0210,0.0074,0.0072,0.0361,0.0668,0.0036,0.0028
140,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,Random,Random_s2026_r3,N/A,N/A,0.000003,...,0.0028,0.0067,0.0172,0.0210,0.0074,0.0072,0.0361,0.0668,0.0036,0.0028
141,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,Random,Random_s2026_r4,N/A,N/A,0.000003,...,0.0028,0.0067,0.0172,0.0210,0.0074,0.0072,0.0361,0.0668,0.0036,0.0028
142,NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_H...,cuda,NVIDIA RTX PRO 6000 Blackwell Server Edition,176,True,Random,Random_s2026_r5,N/A,N/A,0.000005,...,0.0028,0.0067,0.0172,0.0210,0.0074,0.0072,0.0361,0.0668,0.0036,0.0028


### 8. Consolidación y Exportación de Resultados
**Decisión técnica:** Se transforma la lista de registros en un DataFrame de Pandas para su manipulación tabular. El archivo final se nombra dinámicamente utilizando la etiqueta del hardware detectado en el Paso 4. Finalmente, se gatilla la descarga automática del CSV, entregando el artefacto listo para la etapa de análisis visual y estadístico del estudio.

In [8]:
from google.colab import files

output_filename = f"resultados_{HW_LABEL}.csv"
results_df.to_csv(output_filename, index=False)

print(f"archivo : {output_filename}")
print(f"filas   : {len(results_df)}")
print(f"columnas: {len(results_df.columns)}")
print()
print(results_df.groupby("model")[["co2_kg", "time_s"]].mean().round(8))

files.download(output_filename)

archivo : resultados_NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition_HighRAM.csv
filas   : 144
columnas: 23

            co2_kg    time_s
model                       
BPR       0.000010  1.390556
FM        0.000026  2.245000
ItemKNN   0.000012  1.475556
LightGCN  0.000019  1.741667
NGCF      0.000022  1.828889
NeuMF     0.000019  1.690556
Pop       0.000007  0.914444
Random    0.000004  0.716111


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>